# Multipath as measurements: predict the paths, pick them out, count what is usable

The receiver design of `processing/notes/D2_multipath.md`, tried on the phase 1 pinger recordings of the
two buoys, without the tracker:

1. **Predict every path from the bathymetry.** Delay, launch and arrival angle, and Doppler of each surface
   and bottom path, ray-traced over the EMODnet seabed (`processing/multipath_predict.py`).
2. **Compensate Doppler first**, then matched-filter.
3. **Use the geometry for windows, not for a replica.** Free sparse path extraction (OMP) on the compressed pulse;
   the predicted delays only set the search window of each path family.
4. **The direct path is the earliest arrival above the false-alarm level** inside the window the track predicts,
   not the biggest peak.
5. **Associate peaks with paths**, then keep a measurement only if the association is unique, the path is
   resolved, and it persists from ping to ping.

The end of the notebook answers: **from how many chirps can multipath be used, and what are the measurements.**
The computation is `processing/multipath_paths.py` (output `out/multipath/paths_phase1.csv`); this notebook
re-runs it from its caches and plots it. Everything follows `processing/forfun.toml` and the pipeline outputs, and
`processing/pipeline.py` re-executes the notebook when either changes. The text quotes the numbers of the
2026-10-22 run.

**Unknowns, as placeholders** (`multipath_predict.py`): the pinger depth (configured 3 m; windows span 1–5 m), the
hydrophone depth (10 m above the seabed: 3.28 m / 2.65 m with the tide; windows span ±2 m) and the surge on top of the predicted tide (`[site] tide`,
+0.02 m on 2026-10-20; windows span ± `tide_sigma` = 0.2 m). The sound speed is the phase 1 posterior,
isovelocity (no profile is known).

## Findings

**Answer: 19 of the 740 chirps (2.6 %) give a usable multipath delay, 18 of them on Rx01. No chirp gives two path
pairs. The usable bottom bounces arrive earlier than the configured pinger depth predicts: they want a pinger
deeper than 3 m (the edge of the 1–5 m grid).**
(Run of 2026-10-22: hydrophones 10 m above the seabed, i.e. 3.28 m (Rx902) and 2.65 m (Rx01) deep with the
2026-10-20 tide; data-derived run times.)

- **Prediction.** The resolution is 1/B = 0.1 ms (15 cm). With both the pinger (3 m) and the hydrophones (~3 m)
  near the surface, the surface path S arrives only 0.01–0.04 ms after the direct path, never separable. The
  bottom path B is now 0.06–0.31 ms behind (above 1/B on 83–86 % of pings), SB 0.23–0.61 ms and BS 0.09–0.50 ms.
  The unknown depths make the windows 0.04–0.36 ms wide, and B, SB and BS overlap.
- **Model choice.** The flat-bottom image method is 40–120 µs off for the bottom paths, more than 1/B, so the
  sloping seabed matters. The cross-path slope (≤ 1.2°) changes B by at most 0.08 µs: no 3D model needed.
  Isovelocity assumed; a 0.05–0.2 m/s per m gradient would move the delays by 5–160 µs.
- **Doppler.** 0.26 ms per m/s coupling with the 100 ms chirp; up to 1.3 ms bias without Doppler matching, at most
  4 ns per path after the common time scale.
- **Direct path.** A reflection is stronger than the direct path on 28 % (Rx902) and 42 % (Rx01) of pings (median
  3 dB). The earliest component above the false-alarm level finds the direct path on 97 % / 94 % of pings.
- **Association.** 1655 (Rx902) and 965 (Rx01) components pass the level. Unique: S 30 + 35, B 11 + 22,
  SB 17 + 37, BS 0 (BS always shares its window with B or SB). After the resolution and persistence tests,
  1 + 18 usable delays remain (Rx01: B 10, SB 6, S 2). The 2 "S" delays (104–109 µs) sit in the window margin, far
  from the 22 µs S prediction, so they are probably not S. On Rx902, 563 components arrive after every window,
  including the persistent late arrival near 0.77 ms: no path with at most two reflections reaches it now.
- **Quality and consistency.** Too few consecutive pings for a ping-to-ping scatter, so σ falls back to one
  resolution cell (29 µs). The 10 Rx01 bottom bounces measure 170–188 µs against 306 µs predicted at the
  configured depths (window 176–461 µs). The best common geometry is a 5 m pinger and a 4.6 m hydrophone, both
  at the deep edge of the grid (median misfit 10 µs, against 117 µs at the configured depths). SB alone fits
  the configured depths to 10 µs. **The bottom bounces point to a pinger deeper than 3 m**; the 1–5 m pinger
  grid is too narrow to say how deep.
- **What this means for a tracker.** About 19 delays in 740 chirps, mostly bottom bounces on Rx01 at the start of
  run 1. They are informative about the pinger depth, not yet about the track.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "processing"))

import config
import detect
import multipath_paths as mpaths
import multipath_predict as mpp
import multipath_range as mp
import recordings

cfg = config.load()
geo = mpp.geometry(cfg)
T, C = geo["T"], geo["c"]
per, M = mpaths.measurements(cfg, verbose=False)   # cached predictions and components
FAMS = mpp.FAMS
FCOL = {f: f"C{i}" for i, f in enumerate(FAMS)}   # one fixed colour per path family
RES = mpaths.resolution(cfg)
TIDES = mpp.tide_offsets(cfg)                      # water-level offsets of the window grid about the configured tide
t_emit = geo["t_emit"]
col = {n: list(T["receivers"]).index(n) for n in mp.BUOYS}
print(f"sound speed {C:.1f} m/s (phase 1), resolution 1/B = {RES * 1e3:.2f} ms = {RES * C * 100:.0f} cm of path")
print(f"configured depths: pinger {cfg.pinger.depth:g} m, hydrophones "
      + ", ".join(f"{n} {cfg.receiver(n).depth:g} m" for n in mp.BUOYS))

## 1. Predicting the paths

For every ping, the bottom profile from the track position to each buoy (EMODnet, 10 m steps) and straight
isovelocity rays over it (`bathy.eigenrays`, specular reflection on each sloping facet; at most two reflections:
S, B, SB, BS). Solid: ray-traced at the configured depths. Shaded: each family's **search window**, the range of
its delay over the depth and tide grid. Dashed: the flat-bottom image method at the mean depth of the profile.
Grey line: the resolution 1/B. A path is separable from the direct one only above it.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(15, 8), sharex=True)
for ax, n in zip(axs, mp.BUOYS):
    p = per[n]["pred"]
    o = np.argsort(t_emit)
    for q, f in enumerate(FAMS):
        ax.fill_between(t_emit[o], p["lo"][o, q] * 1e3, p["hi"][o, q] * 1e3, color=FCOL[f], alpha=0.15, lw=0)
        ax.plot(t_emit[o], p["delay"][o, q] * 1e3, color=FCOL[f], lw=1.5, label=f"{f}: ray-traced")
        ax.plot(t_emit[o], p["image"][o, q] * 1e3, color=FCOL[f], lw=1, ls="--")
    ax.axhline(RES * 1e3, color="0.4", lw=1)
    ax.text(t_emit.min(), RES * 1e3, " 1/B", va="bottom", fontsize=8, color="0.3")
    ax.set_ylim(0, 1.0)
    ax.set_ylabel("delay after the direct path [ms]")
    ax.set_title(f"{n}: predicted delays (shaded: window over pinger depth {mpp.ZS_GRID[0]:g}-{mpp.ZS_GRID[-1]:g} m, "
                 f"hydrophone {mpp.ZR_DELTA[0]:+g}..{mpp.ZR_DELTA[-1]:+g} m, water level ±{TIDES[-1]:g} m; dashed: flat-bottom image method)",
                 fontsize=9, loc="left")
    ax.legend(ncol=4, fontsize=8, loc="upper right")
axs[-1].set_xlabel("emission time, buoy clock [s]")
fig.tight_layout()
for n in mp.BUOYS:
    p = per[n]["pred"]
    print(f"{n}: range {np.nanmin(p['R']):.0f}-{np.nanmax(p['R']):.0f} m, depth under the pinger "
          f"{np.nanmin(p['D_src']):.1f}-{np.nanmax(p['D_src']):.1f} m")
    for q, f in enumerate(FAMS):
        w = (p["hi"][:, q] - p["lo"][:, q]) * 1e3
        print(f"   {f:3s} nominal {np.nanmin(p['delay'][:, q]) * 1e3:.3f}-{np.nanmax(p['delay'][:, q]) * 1e3:.3f} ms, "
              f"window width median {np.nanmedian(w):.2f} ms, below 1/B on {np.nanmean(p['delay'][:, q] < RES) * 100:3.0f} % of pings")

### Is the image method enough? Is a 3D model needed?

The note's rule: the image method is enough for a flat or gently sloping, well-mixed site; otherwise
precomputed BELLHOP tables, and Bellhop3D where slopes cross the track. Neither BELLHOP nor arlpy is used here
(plain numpy). The two questions can be answered with geometry alone:

- **Along-path slope (2D):** difference between the ray-traced delays (sloping facets) and the flat-bottom image
  method, compared with 1/B.
- **Cross-path slope (3D):** at the bounce point of the bottom path B, the exact image source in the local
  seabed plane, with and without the slope across the path. The difference is the delay error of an Nx2D model.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
for n, ls in zip(mp.BUOYS, ("-", "--")):
    p = per[n]["pred"]
    o = np.argsort(t_emit)
    for q, f in enumerate(FAMS):
        axs[0].plot(t_emit[o], np.abs(p["delay"][o, q] - p["image"][o, q]) * 1e6, color=FCOL[f], ls=ls, lw=1,
                    label=f"{f}, {n}")
    axs[1].plot(t_emit[o], p["slope_along"][o], color="k", ls=ls, lw=1, label=f"along, {n}")
    axs[1].plot(t_emit[o], p["slope_across"][o], color="C4", ls=ls, lw=1, label=f"across, {n}")
    axs[2].plot(t_emit[o], np.abs(p["dtau3d"][o]) * 1e6, color="k", ls=ls, lw=1, label=n)
axs[0].axhline(RES * 1e6, color="0.4", lw=1)
axs[0].set_yscale("log")
axs[0].set_ylabel("|ray-traced - image method| [µs]")
axs[0].set_title("2D: sloping vs flat bottom (grey: 1/B)", fontsize=9)
axs[0].legend(ncol=2, fontsize=7)
axs[1].set_ylabel("seabed slope at the B bounce [deg]")
axs[1].set_title("slope along and across the path", fontsize=9)
axs[1].legend(fontsize=7)
axs[2].set_yscale("log")
axs[2].set_ylabel("|delay change of B| [µs]")
axs[2].set_title("3D: cross-path slope included vs not", fontsize=9)
axs[2].legend(fontsize=7)
for ax in axs:
    ax.set_xlabel("emission time [s]")
fig.tight_layout()
for n in mp.BUOYS:
    p = per[n]["pred"]
    d2 = np.abs(p["delay"] - p["image"])
    print(f"{n}: |ray - image| 95th percentile " + ", ".join(f"{f} {np.nanpercentile(d2[:, q], 95) * 1e6:.0f} µs"
                                                         for q, f in enumerate(FAMS))
          + f"; cross-path slope up to {np.nanmax(np.abs(p['slope_across'])):.1f} deg, "
            f"3D delay change up to {np.nanmax(np.abs(p['dtau3d'])) * 1e6:.2f} µs")

### Angles and Doppler of each path

The pinger moves; the buoys do not. Each path leaves the pinger at its own vertical launch angle θ, so it sees
the closing speed $v_r\cos\theta$ and the Doppler $f_c v_r \cos\theta / c$ ($f_c$ = 35 kHz). The arrival angle is
the direction of travel at the hydrophone, positive down. It would gate the association on an array; the
buoys have one hydrophone, so it is listed but not used.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 4.2))
o = np.argsort(t_emit)
for n, ls in zip(mp.BUOYS, ("-", "--")):
    p = per[n]["pred"]
    for j, f in enumerate(mpp.PATHS):
        c_ = "k" if f == "D" else FCOL[f]
        axs[0].plot(t_emit[o], p["launch"][o, j], color=c_, ls=ls, lw=1, label=f"{f}, {n}")
        axs[1].plot(t_emit[o], p["arrival"][o, j], color=c_, ls=ls, lw=1)
        if j:
            axs[2].plot(t_emit[o], np.abs(p["doppler"][o, j] - p["doppler"][o, 0]), color=c_, ls=ls, lw=1)
axs[0].set_ylabel("launch angle at the pinger [deg, + down]")
axs[1].set_ylabel("arrival angle at the buoy [deg, + down]")
axs[2].set_ylabel("|Doppler of the path - Doppler of D| [Hz]")
axs[2].set_yscale("log")
axs[0].legend(ncol=2, fontsize=7)
for ax in axs:
    ax.set_xlabel("emission time [s]")
fig.tight_layout()
for n in mp.BUOYS:
    p = per[n]["pred"]
    print(f"{n}: closing speed {np.nanmin(p['v_r']):+.2f}..{np.nanmax(p['v_r']):+.2f} m/s, common Doppler "
          f"{np.nanmin(p['doppler'][:, 0]):+.0f}..{np.nanmax(p['doppler'][:, 0]):+.0f} Hz; differential Doppler up to "
          + ", ".join(f"{f} {np.nanmax(np.abs(p['doppler'][:, j + 1] - p['doppler'][:, 0])):.2f} Hz" for j, f in enumerate(FAMS)))

## 2. Doppler first

The LFM's range–Doppler coupling moves the matched-filter peak by $T f_1/B$ per unit of time-scale error. The note's
0.23 ms at 2 m/s assumed a 50 ms chirp. This pinger's chirp is 100 ms, so the coupling is twice that:
`detect.coupling` = 0.4 s per unit, i.e. **0.26 ms per m/s**.

The compressed pulses used here (`multipath_range.extract`) are already Doppler-matched. The replica is
time-scaled by η, estimated from the receiver's own arrival sequence (`detect.eta_track`: the ping interval is
$T_p/\eta$). That is the same as resampling the signal by 1/η, and it does not depend on the multipath. One
common scale leaves each path the residual bias of its own $\cos\theta$, computed below. The example ping is the
fastest-closing one, filtered both ways.

In [ ]:
budget = {n: mpp.doppler_budget(per[n]["pred"], cfg, C) for n in mp.BUOYS}
for n in mp.BUOYS:
    b = budget[n]
    print(f"{n}: bias without Doppler matching |median| {np.nanmedian(np.abs(b['common'])) * 1e3:.2f} ms, "
          f"max {np.nanmax(np.abs(b['common'])) * 1e3:.2f} ms ({np.nanmax(np.abs(b['common'])) / RES:.1f} x 1/B); "
          f"residual per path after one common scale: max "
          + ", ".join(f"{f} {np.nanmax(np.abs(b['residual'][:, q])) * 1e9:.0f} ns" for q, f in enumerate(FAMS)))


def compressed(n, k, eta):
    # matched-filter envelope of ping row k on buoy n with time scale eta, times rel. the detected TOA
    D_ = np.load(ROOT / "out" / f"detections_{n}.npz")
    i = T["det_index"][k, col[n]]
    t_rx = D_["det"]["t_rx"][i]
    rec = recordings.load(cfg.receiver(n))
    mf = detect.MatchedFilter(cfg.pinger, rec.fs, int(0.25 * rec.fs), shapes=D_["shapes"])
    x, tx0 = rec.segment(t_rx - 0.01, t_rx + 0.03 + 1.05 * cfg.pinger.duration, clock="rx")
    c_ = mf.out(mf.xspec(x[:, :1]), eta)[:, 0]
    return tx0 + np.arange(len(c_)) / rec.fs - t_rx, np.abs(c_)


n_ex = "Rx902"
ok = mp.extract(n_ex)["ok"] & (per[n_ex]["direct"] >= 0)
k_fast = int(np.flatnonzero(ok)[np.nanargmax(np.abs(per[n_ex]["pred"]["v_r"][ok]))])
eta_k = np.load(ROOT / "out" / f"detections_{n_ex}.npz")["det"]["eta"][T["det_index"][k_fast, col[n_ex]]]
fig, ax = plt.subplots(figsize=(14, 4))
for eta, lab, c_ in ((1.0, "zero-Doppler replica (η = 1)", "0.6"), (eta_k, f"Doppler-matched (η = {eta_k:.5f})", "k")):
    t, a = compressed(n_ex, k_fast, eta)
    m = (t > -1.5e-3) & (t < 2.5e-3)
    ax.plot(t[m] * 1e3, 20 * np.log10(a[m] / a[m].max()), color=c_, lw=1.2, label=lab)
    print(f"{lab}: peak at {t[m][np.argmax(a[m])] * 1e3:+.3f} ms")
pred_shift = detect.coupling(cfg.pinger) * (1.0 - eta_k)
ax.axvline(pred_shift * 1e3, color="C3", lw=1, ls=":", label=f"predicted coupling shift {pred_shift * 1e3:+.2f} ms")
ax.set_ylim(-30, 2)
ax.set_xlabel("time after the detected TOA [ms]")
ax.set_ylabel("matched-filter output [dB]")
ax.set_title(f"{n_ex}, ping {T['k'][k_fast]}: closing at {per[n_ex]['pred']['v_r'][k_fast]:+.2f} m/s", loc="left", fontsize=9)
ax.legend(fontsize=8)
fig.tight_layout()

## 3. Windows, path extraction and the direct path

**Extraction.** Orthogonal matching pursuit (`multipath_range.sparse_paths`, up to 15 components) on the
Doppler-matched compressed pulse over −0.6…9 ms. The model of one path is the measured pulse
shape. Its own sidelobes, at ±0.14 ms and −8 to −10 dB, are part of the model, so they are not extra paths. Amplitudes and phases
are fitted, never predicted: at 35 kHz a quarter wavelength is 1 cm, far below what the geometry can predict.

**False-alarm level.** No path can arrive before the direct one. Whatever the extractor finds there (more than
1/B before the direct-path window) is noise or pulse-shape mismatch. Its 95th percentile, relative to the
strongest component of the ping, is the level every path must exceed.

**Direct path.** The earliest component above that level within ±1/B of the TOA the phase 1 track predicts for
this buoy (`state.npz: toa_pred`). It is not the biggest peak.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
for ax, n in zip(axs, mp.BUOYS):
    p = per[n]
    after = (p["cmp"]["tau"] > p["centre"][p["cmp"]["row"]] + RES)
    bins = np.arange(-40, 1, 1.0)
    ax.hist(p["rel"][p["early"]], bins, color="0.5", alpha=0.7, density=True, label="before the direct window (false alarms)")
    ax.hist(p["rel"][after], bins, histtype="step", color="k", density=True, label="after it")
    ax.axvline(p["thr"], color="C3", lw=1.5, label=f"false-alarm level {p['thr']:.1f} dB")
    ax.set_xlabel("component level rel. the strongest of its ping [dB]")
    ax.set_title(n, loc="left", fontsize=9)
    ax.legend(fontsize=8)
fig.tight_layout()
for n in mp.BUOYS:
    p = per[n]
    has = p["direct"] >= 0
    n_ok = int(mp.extract(n)["ok"].sum())
    ns = has & (p["direct"] != p["strongest"])
    lvl_d = p["rel"][p["direct"][has]]
    off = p["cmp"]["tau"][p["direct"][has]]
    print(f"{n}: direct path found on {has.sum()} of {n_ok} pings; it is not the strongest component on {ns.sum()} "
          f"({100 * ns.sum() / has.sum():.0f} %), median {np.median(lvl_d[lvl_d < 0]):.1f} dB below it; "
          f"more than 1/B from detect.py's TOA on {np.sum(np.abs(off) > RES)}")

### Three pings, close up

Envelope of the compressed pulse (grey), the extracted components (stems: black = direct; coloured = assigned to a
family; filled = usable), the false-alarm level (red), the direct window (hatched) and the family windows
(shaded). The pings are the first with two usable families, a ping where a reflection is stronger than the
direct path, and a typical ping on Rx01.

In [ ]:
def show_ping(ax, n, k):
    p, d = per[n], mp.extract(n)
    A = np.abs(d["y"][k]) ** 2
    iD = p["direct"][k]
    tD = p["cmp"]["tau"][iD]
    top = 10 * np.log10(A.max())
    ax.plot((d["t"] - tD) * 1e3, 10 * np.log10(A + 1e-30) - top, color="0.7", lw=1)
    for q, f in enumerate(FAMS):
        lo, hi = p["pred"]["lo"][k, q], p["pred"]["hi"][k, q]
        if np.isfinite(lo):
            ax.axvspan(lo * 1e3, hi * 1e3, ymin=0.9 - 0.05 * q, ymax=0.95 - 0.05 * q, color=FCOL[f], alpha=0.5)
            ax.text(hi * 1e3, -2.5 - 1.6 * q, f" {f}", color=FCOL[f], fontsize=7, va="center")
            ax.axvline(p["pred"]["delay"][k, q] * 1e3, color=FCOL[f], lw=0.8, ls=":")
    c0 = p["centre"][k] - tD
    ax.axvspan((c0 - RES) * 1e3, (c0 + RES) * 1e3, color="none", hatch="///", ec="0.6", lw=0)
    ax.axhline(p["thr"], color="C3", lw=1)
    rows = np.flatnonzero(p["cmp"]["row"] == k)
    mk = M[(M["buoy"] == n) & (M["row"] == k)]
    for i in rows:
        dt = (p["cmp"]["tau"][i] - tD) * 1e3
        lv = p["rel"][i]
        hit = mk[np.isclose(mk["dtau"] * 1e3, dt)]
        if i == iD:
            c_, fill = "k", True
        elif len(hit) and "|" not in hit["family"][0]:
            c_, fill = FCOL[hit["family"][0]], bool(hit["usable"][0])
        else:
            c_, fill = "0.45", False
        ax.plot([dt, dt], [-40, lv], color=c_, lw=1)
        ax.plot(dt, lv, "o", ms=6, color=c_, mfc=c_ if fill else "white")
    ax.set_xlim(-0.6, 2.0)
    ax.set_ylim(-40, 1)
    ax.set_ylabel("power rel. the peak [dB]")
    ax.set_title(f"{n}, ping {T['k'][k]} (t = {t_emit[k]:.0f} s, R = {p['pred']['R'][k]:.0f} m)", loc="left", fontsize=9)


u = M[M["usable"]]
two = [r for r in np.unique(u["row"]) if len(np.unique(u["family"][u["row"] == r])) >= 2]
nb = [(n, r) for n in mp.BUOYS for r in two if ((u["buoy"] == n) & (u["row"] == r)).sum() >= 2]
ex = [nb[0] if nb else (u["buoy"][0], u["row"][0])]
p9 = per["Rx902"]
stronger = np.flatnonzero((p9["direct"] >= 0) & (p9["direct"] != p9["strongest"]))
ex.append(("Rx902", int(stronger[np.argmax(p9["rel"][p9["direct"][stronger]] * -1)])) if len(stronger) else ex[0])
ok01 = np.flatnonzero(per["Rx01"]["direct"] >= 0)
ex.append(("Rx01", int(ok01[len(ok01) // 2])))
fig, axs = plt.subplots(len(ex), 1, figsize=(14, 3.3 * len(ex)), sharex=True)
for ax, (n, k) in zip(axs, ex):
    show_ping(ax, n, int(k))
axs[-1].set_xlabel("delay after the direct path [ms]")
fig.tight_layout()

## 4. Association across the run

Each component above the false-alarm level after the direct path is assigned to the family whose window holds it.
It becomes a measurement only if all three hold:

- **Unique:** exactly one family window holds it, and that window holds no other component.
- **Resolved:** at least 1/B after the direct path, and at least 1/(2B) from any other component.
- **Persistent:** the same family is found at the predicted delay change (±30 µs) on at least 2 of the ±2
  neighbouring pings.

Grey dots: every component above the level. Coloured: components assigned to a single family. Filled: usable.

In [ ]:
fig, axs = plt.subplots(2, 1, figsize=(15, 9), sharex=True)
for ax, n in zip(axs, mp.BUOYS):
    p = per[n]
    rows, tau = p["cmp"]["row"], p["cmp"]["tau"]
    has = p["direct"][rows] >= 0
    dt = np.where(has, tau - tau[np.maximum(p["direct"][rows], 0)], np.nan)
    m = has & (p["rel"] >= p["thr"]) & (dt > 0)
    ax.scatter(t_emit[rows[m]], dt[m] * 1e3, s=4, color="0.75", lw=0, label="component above the level")
    o = np.argsort(t_emit)
    for q, f in enumerate(FAMS):
        ax.fill_between(t_emit[o], p["pred"]["lo"][o, q] * 1e3, p["pred"]["hi"][o, q] * 1e3, color=FCOL[f], alpha=0.12, lw=0)
        mf = M[(M["buoy"] == n) & (M["family"] == f)]
        ax.scatter(t_emit[mf["row"]], mf["dtau"] * 1e3, s=10, facecolors="none", edgecolors=FCOL[f], lw=0.7)
        mu = mf[mf["usable"]]
        ax.scatter(t_emit[mu["row"]], mu["dtau"] * 1e3, s=22, color=FCOL[f], label=f"{f}: usable ({len(mu)})")
    ax.axhline(RES * 1e3, color="0.4", lw=1)
    ax.set_ylim(0, 2.0)
    ax.set_ylabel("delay after the direct path [ms]")
    ax.set_title(n, loc="left", fontsize=9)
    ax.legend(fontsize=8, loc="upper right", ncol=3)
axs[-1].set_xlabel("emission time [s]")
fig.tight_layout()

print(f"{'':6s} {'family':6s} {'in window':>9s} {'unique':>7s} {'+resolved':>9s} {'+persistent':>11s}")
for n in mp.BUOYS:
    m = M[M["buoy"] == n]
    for f in FAMS:
        inw = np.array([f in s.split("|") for s in m["family"]])
        one = m["family"] == f
        print(f"{n:6s} {f:6s} {inw.sum():9d} {(one & m['unique']).sum():7d} {(one & m['unique'] & m['resolved']).sum():9d} "
              f"{(one & m['usable']).sum():11d}")
    p = per[n]
    rows, tau = p["cmp"]["row"], p["cmp"]["tau"]
    has = p["direct"][rows] >= 0
    dt = np.where(has, tau - tau[np.maximum(p["direct"][rows], 0)], np.nan)
    m_ = has & (p["rel"] >= p["thr"]) & (dt > 0)
    beyond = m_ & (dt > np.nanmax(p["pred"]["hi"], 1)[rows] + RES / 2)
    print(f"{n:6s} components above the level after the direct path: {m_.sum()}, within 1/B of it {np.sum(m_ & (dt < RES))}, "
          f"later than every window {beyond.sum()}")

## 5. Are the measurements consistent with one geometry?

There is no ground truth, so two checks come from the data itself:

- **Ping-to-ping scatter** of each usable family's delay, net of the predicted change, against the CRLB. This
  gives the measurement σ.
- **One common geometry.** If the assigned paths really are S, B, SB and BS, then a single pinger depth,
  hydrophone depth and tide should predict all of their delays to within that σ. The map shows the median
  |measured − predicted| over the depth grid (at the best tide), for all usable delays together and for each family
  alone. The grid steps are 1 m, so a few µs of the misfit are grid quantisation.

In [ ]:
for n in mp.BUOYS:
    print(n + ": " + ", ".join(f"{f} ping-to-ping {per[n]['sigma_emp'][f] * 1e6:.1f} µs" for f in FAMS
                               if np.isfinite(per[n]["sigma_emp"][f]))
          + f"; median CRLB {np.median(M['sigma_crlb'][(M['buoy'] == n) & M['usable']]) * 1e6:.2f} µs")
sets = [("all families", M)] + [(f"{f} only", M[M["family"] == f]) for f in FAMS if np.any(M["usable"] & (M["family"] == f))]
fig, axs = plt.subplots(len(mp.BUOYS), len(sets), figsize=(4.8 * len(sets), 8.5), squeeze=False, layout="constrained")
for c_, (lab, Ms) in enumerate(sets):
    fit = mpaths.depth_fit(Ms, per)
    for r_, n in enumerate(mp.BUOYS):
        e, cnt = fit[n]
        ax = axs[r_, c_]
        if cnt == 0:
            ax.set_axis_off()
            ax.set_title(f"{n}, {lab}: no usable delays", fontsize=9)
            continue
        i = np.unravel_index(np.nanargmin(e), e.shape)
        zr0 = cfg.receiver(n).depth
        im = ax.imshow(e[:, :, i[2]].T * 1e6, origin="lower", aspect="auto", cmap="viridis_r", vmin=0, vmax=150,
                       extent=[mpp.ZS_GRID[0] - 0.5, mpp.ZS_GRID[-1] + 0.5, zr0 + mpp.ZR_DELTA[0] - 0.5, zr0 + mpp.ZR_DELTA[-1] + 0.5])
        ax.plot(mpp.ZS_GRID[i[0]], zr0 + mpp.ZR_DELTA[i[1]], "r+", ms=12)
        ax.plot(cfg.pinger.depth, zr0, "wo", mfc="none", ms=9)
        ax.set_title(f"{n}, {lab}: {cnt} delays\nbest {np.nanmin(e) * 1e6:.0f} µs at zs {mpp.ZS_GRID[i[0]]:g} m, "
                     f"zr {zr0 + mpp.ZR_DELTA[i[1]]:g} m, water level {TIDES[i[2]]:+g} m", fontsize=9)
        ax.set_xlabel("pinger depth [m]")
        ax.set_ylabel("hydrophone depth [m]")
        print(f"{n:6s} {lab:13s} n {cnt:3d}: best {np.nanmin(e) * 1e6:5.1f} µs (zs {mpp.ZS_GRID[i[0]]:g}, zr {zr0 + mpp.ZR_DELTA[i[1]]:g}, "
              f"water level {TIDES[i[2]]:+g}); at the configured depths and tide {e[np.argmin(np.abs(mpp.ZS_GRID - cfg.pinger.depth)), np.argmin(np.abs(mpp.ZR_DELTA)), np.argmin(np.abs(TIDES))] * 1e6:5.1f} µs")
fig.colorbar(im, ax=axs, label="median |measured - predicted| [µs]  (+ best, o configured)", shrink=0.8)

## 6. Result: how many chirps, and the measurement list

A **measurement** is one delay difference $\Delta\tau = \tau_\text{path} - \tau_\text{direct}$ on one buoy for one
ping. It needs no clock sync and no emission time. Its model is the eigenray delay difference for the pinger
position, $\Delta\tau_f(\mathbf{p}, z_s, z_r, \text{tide}, c)$. σ is the CRLB combined with the family's
ping-to-ping scatter. The full list (every assigned component, with flags) is
`out/multipath/paths_phase1.csv`. The usable rows are printed below.

In [ ]:
n_pings = len(T["k"])
u = M[M["usable"]]
fams_per_ping = {}
for r, n, f in zip(u["ping"], u["buoy"], u["family"]):
    fams_per_ping.setdefault(r, set()).add((n, f))
n_any = len(fams_per_ping)
n_two = sum(len(v) >= 2 for v in fams_per_ping.values())
n_two_same = sum(any(sum(b == n for b, _ in v) >= 2 for n in mp.BUOYS) for v in fams_per_ping.values())
print(f"pings (chirps) in phase 1: {n_pings}")
for n in mp.BUOYS:
    un = u[u["buoy"] == n]
    print(f"  {n}: {len(un)} usable delays on {len(np.unique(un['ping']))} pings  ("
          + ", ".join(f"{f} {np.sum(un['family'] == f)}" for f in FAMS) + ")")
print(f"chirps with >= 1 usable multipath delay: {n_any} ({100 * n_any / n_pings:.1f} %)")
print(f"chirps with >= 2 (either buoy): {n_two}; with two path pairs on the same buoy (range and depth from one buoy): {n_two_same}")
print(f"bottom-only paths (B) usable: {np.sum(u['family'] == 'B')}")
print()
print(f"{'ping':>5s} {'run':>3s} {'t_emit [s]':>10s} {'buoy':5s} {'path':4s} {'dtau [µs]':>9s} {'sigma':>6s} {'pred':>6s} "
      f"{'window [µs]':>13s} {'level':>6s} {'SNR':>5s} {'launch':>7s} {'arrival':>7s} {'dDop [Hz]':>9s}")
for r in u[np.lexsort((u["family"], u["buoy"], u["ping"]))]:
    print(f"{r['ping']:5d} {r['run']:3d} {r['t_emit']:10.3f} {r['buoy']:5s} {r['family']:4s} {r['dtau'] * 1e6:9.1f} "
          f"{r['sigma'] * 1e6:6.1f} {r['pred'] * 1e6:6.0f} {r['win_lo'] * 1e6:6.0f}-{r['win_hi'] * 1e6:<6.0f} "
          f"{r['level_db']:6.1f} {r['snr_db']:5.1f} {r['launch_deg']:7.2f} {r['arrival_deg']:7.2f} {r['doppler_hz']:9.3f}")